In [ ]:
pip install selenium pandas openpyxl

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException
)
import pandas as pd
import time

In [ ]:
# Instagram 로그인 함수
def login_to_instagram(driver, username, password):
    driver.get("https://www.instagram.com/")
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "username"))
    )
    driver.find_element("name", "username").send_keys(username)  # 사용자명 입력
    driver.find_element("name", "password").send_keys(password)  # 비밀번호 입력
    driver.find_element("css selector", "button[type='submit']").click()
    time.sleep(5)  # 로그인 처리 대기

In [ ]:
# 해시태그 검색 함수
def search_hashtag(driver, hashtag):
    search_url = f"https://www.instagram.com/explore/tags/{hashtag}/"
    driver.get(search_url)
    time.sleep(5)  # 페이지 로드 대기

In [ ]:
def extract_post_info_likes_hashtag(driver):
    try:
        # 기본 값 설정
        poster_id = "게시자 ID 없음"
        post_text = "게시글 본문 없음"
        comments_count = "0"
        likes = "0"
        views = "0"
        date = "게시 날짜 없음"
        hashtags = []

        # 조회수 추출
        try:
            views_element = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, "//span[contains(text(), '조회')]/span"))
            )
            views = f"조회수: {views_element.text}"
        except TimeoutException:
            print("조회수 없음 또는 조회수 추출 중 타임아웃")
        except Exception as e:
            print("조회수 추출 중 오류:", e)

        # 좋아요 추출
        try:
            # 먼저 '좋아요' 요소를 시도합니다.
            likes_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//span[contains(text(), '좋아요')]/span"))
            )
            likes = f"{likes_element.text}"
        except TimeoutException:
            print("좋아요 요소를 찾지 못했습니다. '여러 명' 또는 다른 상태를 확인합니다.")
            try:
                # '여러 명' 요소를 찾습니다.
                multiple_people_element = driver.find_element(By.XPATH, "//span[text()='여러 명']")
                if multiple_people_element:
                    likes = "게시자 미허용"
            except NoSuchElementException:
                print("여러 명 요소를 찾지 못했습니다.")
                likes = "추출 오류"
            except Exception as e2:
                print("여러 명 요소를 찾는 중 오류:", e2)
                likes = "추출 오류"
        except Exception as e:
            print("좋아요 추출 중 오류:", e)

        # 게시 날짜 추출
        try:
            date_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "time"))
            )
            date = date_element.get_attribute("datetime")
        except TimeoutException:
            print("게시 날짜를 찾는 중 타임아웃")
            date = "게시 날짜 없음"
        except Exception as e:
            print("게시 날짜를 찾는 중 오류:", e)
            date = "게시 날짜 없음"

        # 해시태그 추출
        try:
            hashtag_elements = driver.find_elements(By.XPATH, "//a[contains(@href, '/explore/tags/') and starts-with(text(), '#')]")
            hashtags = [element.text for element in hashtag_elements]
        except Exception as e:
            print("해시태그를 찾는 중 오류:", e)
            hashtags = []

        # 게시자 아이디 추출
        try:
            # 제공된 HTML 구조를 기반으로 한 XPath
            poster_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//h2[contains(@class, 'x6s0dn4')]//a[contains(@href, '/')]"))
            )
            poster_id = poster_element.text
        except TimeoutException:
            print("게시자 아이디를 찾는 중 타임아웃")
        except Exception as e:
            print("게시자 아이디를 찾는 중 오류:", e)

        # 게시글 본문 추출 (해시태그 제외)
        try:
            # 제공된 HTML 구조를 기반으로 한 XPath
            post_text_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//h1[contains(@class, '_ap3a')]"))
            )
            full_text = post_text_element.text
            # 해시태그를 제외한 텍스트 추출
            post_text = ' '.join([word for word in full_text.split() if not word.startswith('#')])
        except TimeoutException:
            print("게시글 본문을 찾는 중 타임아웃")
        except Exception as e:
            print("게시글 본문을 찾는 중 오류:", e)

        return {
            "게시자 ID": poster_id,
            "게시글 본문": post_text,
            "좋아요": likes,
            "조회수": views,
            "게시 날짜": date,
            "해시태그": hashtags
        }

    except WebDriverException as e:
        print("WebDriver 예외 발생:", e)
        return {
            "게시자 ID": "추출 실패",
            "게시글 본문": "추출 실패",
            "좋아요": "추출 실패",
            "조회수": "추출 실패",
            "게시 날짜": "추출 실패",
            "해시태그": []
        }
    except Exception as e:
        print("게시글 정보 추출 중 예기치 않은 오류:", e)
        return {
            "게시자 ID": "추출 실패",
            "게시글 본문": "추출 실패",
            "좋아요": "추출 실패",
            "조회수": "추출 실패",
            "게시 날짜": "추출 실패",
            "해시태그": []
        }


In [ ]:
# 데이터 저장 및 엑셀로 내보내기
def save_to_excel_likes_hashtag(data, filename="instagram_likes_hashtag_data.xlsx"):
    df_likes_hashtag = pd.DataFrame(data)
    df_likes_hashtag.to_excel(filename, index=False)
    print(f"데이터가 {filename}에 저장되었습니다.")

In [ ]:
def wait_for_post_loaded(driver):
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, "//article")  # 게시글이 로드되었는지 확인
            )
        )
        print("게시글 로드 완료")
    except Exception as e:
        print("게시글 로드 중 오류:", e)

In [ ]:
def crawl_posts_likes_hashtag(driver, max_posts):
    posts_data_likes_hashtag = []
    collected_urls = set()

    try:
        # 첫 번째 게시글 클릭
        first_post = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div._aagw"))
        )
        first_post.click()
        print("첫 번째 게시글 클릭 성공")
        time.sleep(3)

        for i in range(max_posts):
            print(f"\n[DEBUG] 게시글 {i + 1} 크롤링 시작...")
            try:
                current_url = driver.current_url
                print(f"[DEBUG] 현재 게시글 URL: {current_url}")

                if current_url in collected_urls:
                    print(f"[DEBUG] 중복된 게시글 {i + 1} 스킵")
                    continue
                collected_urls.add(current_url)

                # 게시글 정보 추출
                post_info_likes_hashtag = extract_post_info_likes_hashtag(driver)
                print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_likes_hashtag}")
                if post_info_likes_hashtag:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_likes_hashtag}")
                    posts_data_likes_hashtag.append(post_info_likes_hashtag)
                else:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터가 비어 있음")
                # 다음 게시글로 이동
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "svg[aria-label='다음']"))
                )
                next_button.click()
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 완료, 다음 게시글로 이동")
                time.sleep(3)
            except Exception as e:
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 중 오류:", e)
                continue
    except Exception as e:
        print("크롤링 중 전체 오류:", e)

    print("\n크롤링 완료")
    return posts_data_likes_hashtag

In [ ]:
def save_to_excel_comments(data, filename="instagram_comments_data.xlsx"):
    # 각 댓글 데이터를 행 단위로 확장할 리스트
    expanded_data = []

    # 데이터가 리스트인 경우 처리
    if isinstance(data, list):
        # 여러 게시글을 포함한 리스트일 경우 각 게시글별로 처리
        for post in data:
            for i in range(len(post["댓글 작성자 ID"])):
                expanded_data.append({
                    "게시자 ID": post["게시자 ID"] if i == 0 else "",  # 첫 번째 행에만 표시
                    "게시물 본문": post["게시물 본문"] if i == 0 else "",  # 첫 번째 행에만 표시
                    "댓글 작성자 ID": post["댓글 작성자 ID"][i] if i < len(post["댓글 작성자 ID"]) else None,
                    "comment": post["comment"][i] if i < len(post["comment"]) else None,
                    "댓글 좋아요 수": post["댓글 좋아요 수"][i] if i < len(post["댓글 좋아요 수"]) else None,
                    "댓글 작성 날짜": post["댓글 작성 날짜"][i] if i < len(post["댓글 작성 날짜"]) else None,
                    "season": post["season"][i] if i < len(post["season"]) else None,
                })
    else:
        # 데이터가 단일 딕셔너리인 경우 처리
        for i in range(len(data["댓글 작성자 ID"])):
            expanded_data.append({
                "게시자 ID": data["게시자 ID"] if i == 0 else "",  # 첫 번째 행에만 표시
                "게시물 본문": data["게시물 본문"] if i == 0 else "",  # 첫 번째 행에만 표시
                "댓글 작성자 ID": data["댓글 작성자 ID"][i] if i < len(data["댓글 작성자 ID"]) else None,
                "comment": data["comment"][i] if i < len(data["comment"]) else None,
                "댓글 좋아요 수": data["댓글 좋아요 수"][i] if i < len(data["댓글 좋아요 수"]) else None,
                "댓글 작성 날짜": data["댓글 작성 날짜"][i] if i < len(data["댓글 작성 날짜"]) else None,
                "season": post["season"][i] if i < len(post["season"]) else None,
            })

    # 확장된 데이터를 DataFrame으로 변환
    df = pd.DataFrame(expanded_data)

    # 엑셀로 저장
    df.to_excel(filename, index=False)
    print(f"데이터가 {filename}에 정확히 저장되었습니다.")

In [ ]:
def scroll_and_load_comments(driver):
    """
    댓글 창을 JavaScript로 스크롤 내리고 "댓글 더 읽어들이기" 버튼을 클릭하여 추가 댓글을 로드합니다.
    버튼이 더 이상 존재하지 않을 때까지 반복합니다.

    :param driver: Selenium WebDriver 인스턴스
    """
    try:
        # 댓글 섹션 요소 찾기
        comment_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul._a9z6._a9za"))
        )

        while True:
            # 댓글 섹션 스크롤 내리기 (JavaScript로 구현)
            driver.execute_script("arguments[0].scrollTop += arguments[0].scrollHeight / 10;", comment_section)
            time.sleep(0.2)  # 부드러운 스크롤을 위해 약간의 대기

            # "댓글 더 읽어들이기" 버튼 클릭
            try:
                # "댓글 더 읽어들이기" 버튼을 기다림
                more_btn = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "svg[aria-label='댓글 더 읽어들이기']"))
                )
                more_btn.click()
                time.sleep(1)  # 버튼 클릭 후 로딩 대기
            except TimeoutException:
                break  # 버튼이 더 이상 없으면 루프 종료
            except Exception as e:
                print(f"'댓글 더 읽어들이기' 버튼 클릭 중 오류 발생: {e}")
                break

    except Exception as e:
        print("[DEBUG] 댓글 섹션 스크롤 및 '댓글 더 읽어들이기' 클릭 중 오류 발생:", e)
        # 디버깅을 위해 스크린샷 찍기

    print("댓글 스크롤 및 '댓글 더 읽어들이기' 작업 완료")

In [ ]:
from datetime import datetime

def get_season(published_date):
    year = published_date.year
    month = published_date.month

    if month in [1, 2]:  # 1월, 2월은 이전 해의 FW
        return f"{year - 1} FW"
    elif month in [3, 4, 5, 6, 7, 8, 9]:  # 3월~9월은 현재 해의 SS
        return f"{year} SS"
    else:  # 8월~10월은 현재 해의 FW
        return f"{year} FW"

In [ ]:
def extract_post_info_comments(driver):
    try:
        # 게시자 정보 추출
        try:
            post_author = driver.find_element(By.XPATH, "//h2[contains(@class, 'x6s0dn4')]//a[contains(@href, '/')]").text.strip()
            post_content = driver.find_element(By.XPATH, "//h1[contains(@class, '_ap3a')]").text.strip()
        except Exception as e:
            post_author = "게시자 ID 없음"
            post_content = "게시물 내용 없음"

        # 댓글 섹션 확인
        comment_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul._a9z6._a9za"))
        )

        # 댓글 작성자 ID 추출
        try:
            commenter_ids = driver.find_elements(By.CSS_SELECTOR, "h3.x6s0dn4.x3nfvp2")
            commenter_ids_text = [commenter.text.strip() for commenter in commenter_ids if commenter.text.strip()]
        except Exception as e:
            commenter_ids_text = []

        # 댓글 내용 추출
        try:
            comment_text_elements = driver.find_elements(By.CSS_SELECTOR, "div._a9zo")
            comment_dates_elements = driver.find_elements(By.CSS_SELECTOR, "a[role='link'] > time._a9ze._a9zf")
            comment_texts_content = []

            if len(comment_text_elements) > 2:
                for idx, element in enumerate(comment_text_elements[1:]):
                    text_content = element.text.strip()
                    for commenter_id in commenter_ids_text:
                        if text_content.startswith(commenter_id):
                            text_content = text_content[len(commenter_id):].strip()
                    try:
                        if idx < len(comment_dates_elements):
                            date_text = comment_dates_elements[idx].text.strip()
                            if date_text in text_content:
                                text_content = text_content.replace(date_text, "").strip()
                    except:
                        pass
                    if "좋아요" in text_content:
                        text_content = text_content.split("좋아요")[0].strip()
                    if "답글 달기" in text_content:
                        text_content = text_content.replace("답글 달기", "").strip()
                    img_elements = element.find_elements(By.CSS_SELECTOR, 'img.x107yiy2.xv8uw2v.x1tfwpuw.x2g32xy')
                    if img_elements:
                        comment_texts_content.append("이미지 댓글")
                    elif text_content:
                        comment_texts_content.append(text_content)
            else:
                comment_texts_content = []
        except Exception as e:
            comment_texts_content = []

        # 댓글 좋아요 수 추출
        try:
            comment_likes_count = []
            comment_blocks = driver.find_elements(By.XPATH, "//div[@class='_a9zr']")
            for block in comment_blocks[1:]:
                try:
                    like_element = block.find_element(By.XPATH, ".//button[@class='_a9ze']//span")
                    like_text = like_element.text.strip()
                    if like_text and "좋아요" in like_text:
                        comment_likes_count.append(like_text.replace("좋아요", "").strip())
                    else:
                        comment_likes_count.append("0")
                except:
                    comment_likes_count.append("0")
        except Exception as e:
            comment_likes_count = ["0"]

        # 댓글 작성 날짜 추출
        try:
            comment_dates_text = [date.get_attribute("datetime") for date in driver.find_elements(By.CSS_SELECTOR, "time._a9ze._a9zf")][1:]
        except Exception as e:
            comment_dates_text = []

        # 시즌 구분 (YYYY FW 또는 YYYY SS)
        comment_seasons = []
        for date in comment_dates_text:
            try:
                year = int(date[:4])  # YYYY-MM-DD에서 YYYY 추출
                month = int(date[5:7])  # MM 추출
                if month in [1, 2]:  # 1월, 2월은 이전 해의 FW
                    season = f"{year - 1} FW"
                elif month in [3, 4, 5, 6, 7, 8, 9]:  # 3월~9월은 현재 해의 SS
                    season = f"{year} SS"
                else:  # 나머지는 현재 해의 FW
                    season = f"{year} FW"
                comment_seasons.append(season)
            except:
                comment_seasons.append("알 수 없음")

        return {
            "게시자 ID": post_author,
            "게시물 본문": post_content,
            "댓글 작성자 ID": commenter_ids_text,
            "comment": comment_texts_content,
            "댓글 좋아요 수": comment_likes_count,
            "댓글 작성 날짜": comment_dates_text,
            "season": comment_seasons
        }

    except WebDriverException as e:
        return {
            "게시자 ID": "오류 발생",
            "게시물 본문": "오류 발생",
            "댓글 작성자 ID": [],
            "comment": [],
            "댓글 좋아요 수": [],
            "댓글 작성 날짜": [],
            "season": []
        }
    except Exception as e:
        return {
            "게시자 ID": "오류 발생",
            "게시물 본문": "오류 발생",
            "댓글 작성자 ID": [],
            "comment": [],
            "댓글 좋아요 수": [],
            "댓글 작성 날짜": [],
            "season": []
        }

In [ ]:
def crawl_posts_comments(driver, max_posts):
    posts_data_comments = []
    collected_urls = set()

    try:
        # 첫 번째 게시글 클릭
        first_post = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div._aagw"))
        )
        first_post.click()
        print("첫 번째 게시글 클릭 성공")
        time.sleep(3)

        for i in range(max_posts):
            print(f"\n[DEBUG] 게시글 {i + 1} 크롤링 시작...")
            try:
                current_url = driver.current_url
                print(f"[DEBUG] 현재 게시글 URL: {current_url}")

                if current_url in collected_urls:
                    print(f"[DEBUG] 중복된 게시글 {i + 1} 스킵")
                    continue
                collected_urls.add(current_url)

                # 게시글 댓글 로딩
                scroll_and_load_comments(driver)

                # 게시글 정보 추출
                post_info = extract_post_info_comments(driver)
                print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info}")
                if post_info:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info}")
                    posts_data_comments.append(post_info)
                else:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터가 비어 있음")
                # 다음 게시글로 이동
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "svg[aria-label='다음']"))
                )
                next_button.click()
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 완료, 다음 게시글로 이동")
                time.sleep(3)
            except Exception as e:
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 중 오류:", e)
                continue
    except Exception as e:
        print("크롤링 중 전체 오류:", e)

    print("\n크롤링 완료")
    return posts_data_comments

In [ ]:
def crawl_posts_likes_hashtag_comments(driver, max_posts):
    posts_data_likes_hashtag = []
    posts_data_comments = []
    collected_urls = set()

    try:
        # 첫 번째 게시글 클릭
        first_post = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div._aagw"))
        )
        first_post.click()
        print("첫 번째 게시글 클릭 성공")
        time.sleep(3)

        for i in range(max_posts):
            print(f"\n[DEBUG] 게시글 {i + 1} 크롤링 시작...")
            try:
                current_url = driver.current_url
                print(f"[DEBUG] 현재 게시글 URL: {current_url}")

                if current_url in collected_urls:
                    print(f"[DEBUG] 중복된 게시글 {i + 1} 스킵")
                    continue
                collected_urls.add(current_url)

                # 게시글 좋아요, 해시태그 추출
                post_info_likes_hashtag = extract_post_info_likes_hashtag(driver)
                print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_likes_hashtag}")
                if post_info_likes_hashtag:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_likes_hashtag}")
                    posts_data_likes_hashtag.append(post_info_likes_hashtag)
                else:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터가 비어 있음")

                # 게시글 댓글 로딩
                scroll_and_load_comments(driver)

                # 게시글 댓글 추출
                post_info_comments = extract_post_info_comments(driver)
                print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_comments}")
                if post_info_comments:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터: {post_info_comments}")
                    posts_data_comments.append(post_info_comments)
                else:
                    print(f"[DEBUG] 게시글 {i + 1} 데이터가 비어 있음")

                # 다음 게시글로 이동
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "svg[aria-label='다음']"))
                )
                next_button.click()
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 완료, 다음 게시글로 이동")
                time.sleep(3)
            except Exception as e:
                print(f"[DEBUG] 게시글 {i + 1} 크롤링 중 오류:", e)
                continue
    except Exception as e:
        print("크롤링 중 전체 오류:", e)

    print("\n크롤링 완료")
    return posts_data_likes_hashtag,posts_data_comments

In [ ]:
def crawling_likes_hashtag():
    # 게시글 크롤링 실행
    posts_data_likes_hashtag = crawl_posts_likes_hashtag(driver, max_posts)

# 데이터 저장
    save_to_excel_likes_hashtag(posts_data_likes_hashtag)

def crawling_comments():
    # 게시글 크롤링 실행
    posts_data_comments = crawl_posts_comments(driver, max_posts)

    # 데이터 저장
    save_to_excel_comments(posts_data_comments)

def crawling_likes_hashtag_comments():
    # 게시글 크롤링 실행
    posts_data_likes_hashtag,posts_data_comments = crawl_posts_likes_hashtag_comments(driver, max_posts)

    # 데이터 저장
    save_to_excel_likes_hashtag(posts_data_likes_hashtag)
    save_to_excel_comments(posts_data_comments)

In [ ]:
def main():
    print("크롤링 데이터 수집방법을 선택하세요:")
    print("1: 좋아요,해시태그")
    print("2: 댓글")
    print("3: 좋아요,해시태그+댓글")

    # 사용자 입력 받기
    choice = input("숫자를 입력하세요 (1, 2, 3): ")

    # 조건에 따라 함수 실행
    if choice == "1":
        crawling_likes_hashtag()
    elif choice == "2":
        crawling_comments()
    elif choice == "3":
        crawling_likes_hashtag_comments()
    else:
        print("잘못된 입력입니다. 1, 2 또는 3을 입력하세요.")

In [ ]:
import os
import platform
from pathlib import Path

driver_path = r"C:\Users\양희원\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"

print("=== 실행 환경 확인 ===")
print("os.name:", os.name)
print("platform:", platform.platform())
print("current working dir:", os.getcwd())
print()

print("=== 경로 문자열 확인 ===")
print("driver_path:", driver_path)
print("abs path:", os.path.abspath(driver_path))
print()

print("=== 파일 존재 여부 확인 ===")
print("os.path.exists  :", os.path.exists(driver_path))
print("os.path.isfile  :", os.path.isfile(driver_path))

p = Path(driver_path)
print("Path.exists()   :", p.exists())
print("Path.is_file()  :", p.is_file())
print("Path.parts      :", p.parts)

In [ ]:
# ChromeDriver 경로 설정
driver_path = r"C:\Users\양희원\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# 로그인 실행
username = "yyhhoo0415"  # Instagram 아이디
password = "@akskdk153"  # Instagram 비밀번호
login_to_instagram(driver, username, password)
time.sleep(4)

# 해시태그 검색 실행
hashtag = input("크롤링할 해시태그를 입력하세요 (예: travel): ")
max_posts = int(input("크롤링할 게시글 개수를 입력하세요: "))
search_hashtag(driver, hashtag)
time.sleep(4)

main()